# BM25 Keyword-Based Search

BM25 is a keyword-based ranking algorithm that scores documents by term frequency and document length normalization. This implementation builds an inverted index for fast exact-match retrieval with efficient and interpretable ranking scores. Output: `data/processed/bm25_index.pkl`

**Workflow:**

1. Install rank_bm25 library
2. Implement BM25Retriever class (`src/bm25.py`)
3. Build BM25 index from corpus
4. Test search on sample queries

## Prerequisites Check

First run `01_data_preparation.ipynb` to generate `corpus.pkl`, `books_sample.parquet`, and `utils.py`.

In [1]:
import os
from pathlib import Path

notebook_dir = Path.cwd()
project_root = notebook_dir.parent if 'notebooks' in str(notebook_dir) else notebook_dir

prereqs = [
    project_root / 'data/processed/corpus.pkl',
    project_root / 'src/utils.py'
]

print("Checking prerequisites:")
for prereq in prereqs:
    exists = os.path.exists(prereq)
    status = "OK" if exists else "MISSING"
    print(f"  {status}: {prereq}")

if all(os.path.exists(p) for p in prereqs):
    print("\nAll prerequisites ready. Continue below.")
else:
    print("\nRun 01_eda.ipynb first!")

Checking prerequisites:
  OK: /home/stoyq/MDS/575/DSCI_575_project_jchuang_esteki/data/processed/corpus.pkl
  OK: /home/stoyq/MDS/575/DSCI_575_project_jchuang_esteki/src/utils.py

All prerequisites ready. Continue below.


## 1. Install Dependencies

Install the rank_bm25 library which implements the Okapi BM25 algorithm for keyword-based document ranking.

In [2]:
import subprocess, sys

try:
    import rank_bm25
    print("rank_bm25 already installed")
except ImportError:
    print("Installing rank_bm25...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rank-bm25", "-q"])
    print("Done")

rank_bm25 already installed


## 2. Create src/bm25.py

Implement the BM25Retriever class. The class tokenizes documents, builds an inverted index using BM25 scoring, and provides search and persistence methods. Document preprocessing uses utilities from src/utils.py to ensure consistency. Output: `src/bm25.py`

In [3]:
from pathlib import Path
import sys

# Get project root
notebook_dir = Path.cwd()
if 'notebooks' in str(notebook_dir):
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

sys.path.insert(0, str(project_root))

# Create directories
(project_root / "src").mkdir(exist_ok=True)
(project_root / "data" / "processed").mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")

Project root: /home/stoyq/MDS/575/DSCI_575_project_jchuang_esteki


In [4]:
bm25_code = '''import pickle
from pathlib import Path
from typing import List, Tuple
import numpy as np
from rank_bm25 import BM25Okapi
from src.utils import tokenize

class BM25Retriever:
    """BM25 keyword-based retriever using Okapi BM25 algorithm."""
    
    def __init__(self):
        """Initialize empty BM25 retriever."""
        self.bm25 = None
        self.corpus = None
    
    def build_index(self, corpus: List[str]):
        """Build BM25 index from corpus."""
        tokenized_corpus = [tokenize(doc) for doc in corpus]
        self.bm25 = BM25Okapi(tokenized_corpus)
        self.corpus = corpus
        print(f"BM25 index built on {len(corpus)} documents")
    
    def search(self, query: str, top_k: int = 5) -> List[Tuple[int, float]]:
        """Search for query in corpus. Returns (doc_id, score) pairs."""
        if self.bm25 is None:
            raise ValueError("Index not built. Call build_index() first.")
        tokenized_query = tokenize(query)
        scores = self.bm25.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(int(idx), float(scores[idx])) for idx in top_indices]
    
    def save(self, path: str):
        """Save BM25 index to disk."""
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'wb') as f:
            pickle.dump(self.bm25, f)
        print(f"BM25 index saved to {path}")
    
    def load(self, path: str):
        """Load BM25 index from disk."""
        with open(path, 'rb') as f:
            self.bm25 = pickle.load(f)
        print(f"BM25 index loaded from {path}")
'''

bm25_file = project_root / "src" / "bm25.py"
with open(bm25_file, 'w') as f:
    f.write(bm25_code)

print("Created src/bm25.py")

Created src/bm25.py


## 3. Build BM25 Index

Load the corpus and construct the BM25 index by tokenizing all documents and computing term-frequency scores. The index enables fast document ranking for subsequent queries. Output: `data/processed/bm25_index.pkl`

In [5]:
import sys
from pathlib import Path

# Make sure project_root is defined (from setup cell)
# If not, add this:
notebook_dir = Path.cwd()
if 'notebooks' in str(notebook_dir):
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

sys.path.insert(0, str(project_root))

from src.bm25 import BM25Retriever
from src.utils import load_corpus

# Use absolute paths
corpus_file = str(project_root / 'data' / 'processed' / 'corpus.pkl')
index_file = str(project_root / 'data' / 'processed' / 'bm25_index.pkl')

print(f"Loading corpus from: {corpus_file}")
corpus = load_corpus(corpus_file)  # Pass absolute path
print(f"Loaded corpus: {len(corpus)} documents\n")

print("Building BM25 index...")
bm25_retriever = BM25Retriever()
bm25_retriever.build_index(corpus)

print(f"\nSaving to: {index_file}")
bm25_retriever.save(index_file)  # Pass absolute path
print("BM25 index complete")

Loading corpus from: /home/stoyq/MDS/575/DSCI_575_project_jchuang_esteki/data/processed/corpus.pkl
Loaded corpus: 20000 documents

Building BM25 index...
BM25 index built on 20000 documents

Saving to: /home/stoyq/MDS/575/DSCI_575_project_jchuang_esteki/data/processed/bm25_index.pkl
BM25 index saved to /home/stoyq/MDS/575/DSCI_575_project_jchuang_esteki/data/processed/bm25_index.pkl
BM25 index complete


## 4. Test BM25 Search

Load the persisted BM25 index and test search functionality on sample queries. Verify that the retriever returns ranked documents with relevance scores for keyword-based queries.

In [6]:
from pathlib import Path
import sys
import pandas as pd

# Make sure project_root is defined
notebook_dir = Path.cwd()
if 'notebooks' in str(notebook_dir):
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

sys.path.insert(0, str(project_root))

from src.bm25 import BM25Retriever

test_queries = ["python programming book", "mystery novel", "cookbook"]

bm25_retriever = BM25Retriever()

# Use absolute path
index_file = str(project_root / 'data' / 'processed' / 'bm25_index.pkl')
bm25_retriever.load(index_file)

# Load data for result enrichment
df = pd.read_parquet(project_root / 'data' / 'processed' / 'books_sample.parquet')
lookup = pd.read_parquet(project_root / 'data' / 'processed' / 'asin_title_lookup.parquet') \
           .set_index('parent_asin')['book_title'].to_dict()

print("BM25 Test Results")
print("="*50)

for query in test_queries:
    results = bm25_retriever.search(query, top_k=3)
    print(f"\nQuery: '{query}'")
    for rank, (doc_id, score) in enumerate(results, 1):
        book_title = lookup.get(df.iloc[doc_id]['parent_asin'], 'Unknown')
        print(f"  {rank}. Doc #{doc_id} - {book_title} - Score: {score:.2f}")

print("\nBM25 retriever working")

BM25 index loaded from /home/stoyq/MDS/575/DSCI_575_project_jchuang_esteki/data/processed/bm25_index.pkl
BM25 Test Results

Query: 'python programming book'
  1. Doc #16530 - Programming the World Wide Web - Score: 15.30
  2. Doc #9463 - The D Programming Language - Score: 14.95
  3. Doc #16534 - Definitive XML Application Development - Score: 14.51

Query: 'mystery novel'
  1. Doc #1922 - Trust Me - Score: 9.19
  2. Doc #3316 - Disappearance of Adèle Bedeau: An Inspector Gorski Investigation - Score: 8.95
  3. Doc #8761 - Seasons - Score: 8.94

Query: 'cookbook'
  1. Doc #13426 - The Homesick Texan Cookbook - Score: 7.31
  2. Doc #290 - The Step-by-Step Instant Pot Cookbook: 100 Simple Recipes for Spectacular Results -- with Photographs of Every Step - Score: 7.07
  3. Doc #5158 - Bean by Bean: A Cookbook: More Than 175 Recipes for Fresh Beans, Dried Beans, Cool Beans, Hot Beans, Savory Beans, Even Sweet Beans - Score: 7.07

BM25 retriever working
